# Data Update

Keeps the local database current. It changes data; its companion `Data Health.ipynb` only reports on it.

- **Prices** come from Yahoo Finance (`pipeline/YF.py`) into `data/prices/`.
- **Fundamentals** come from the SEC's EDGAR service (`pipeline/Edgar.py`, `pipeline/EdgarStatements.py`) into `data/edgar/`.
- **What is tracked** is decided by `pipeline/watchlist-rules.csv`, which you edit by hand. Everything in `data/` is downloaded or generated; do not edit it.

Day to day, run the setup cell and then section 2. Run section 1 only after changing the rules.

In [ ]:
import os, sys
from pathlib import Path
# Work from the project root, wherever this notebook is opened from: every module reads data/ relative to it.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pipeline' / 'watchlist-rules.csv').exists())
os.chdir(root); sys.path.insert(0, str(root))
import pandas as pd

from pipeline.Watchlist import Watchlist
from pipeline.YF import YF
from pipeline.Edgar import Edgar
from pipeline.EdgarStatements import EdgarStatements
from pipeline.DataHealth import DataHealth
from pipeline.EdgarHealth import EdgarHealth

## 1. Build the watchlist from the rules
Run this after editing `pipeline/watchlist-rules.csv` (include by sector, industry or ticker; exclude by ticker only; an optional fourth column names a sub-list). It resolves the rules against the ticker registry, screens any new candidates for liquidity (median daily dollar volume of at least $10M and a price of at least $2), and writes the tickers that pass to `data/watchlist.csv`. Then run section 2 to fetch data for anything new.

In [ ]:
scraper = YF()
previous = set(pd.read_csv(scraper.tickers_file, comment='#', skip_blank_lines=True)['ticker'].astype(str).str.upper())
candidates = Watchlist().resolve()
screen = scraper.screen(candidates)
admitted = screen.loc[screen['passed'], 'ticker']
if admitted.empty:
    raise RuntimeError("No tickers passed the screen; the watchlist was left unchanged.")
scraper._write_csv_atomic(admitted.to_frame(), scraper.tickers_file)
print(f"{len(candidates)} candidates -> {len(admitted)} admitted -> {scraper.tickers_file}")
print("Dropped vs previous watchlist:", sorted(previous - set(admitted)))
print("Added vs previous watchlist:", sorted(set(admitted) - previous))
print("Sub-lists:", Watchlist().lists().to_dict())

## 2. Update everything
**Prices:** fetches new prices up to seven days ago (so no part-finished trading day is ever stored), refreshes any ticker whose adjusted prices have changed after a dividend or split, and compares the full history of 8 tickers with Yahoo, repairing any that differ.

**Fundamentals:** fetches SEC filings for any ticker not fetched in the last 7 days and rebuilds the quarterly and annual statement files (about 40 seconds).

It ends with the two health summaries. An empty table under a summary means every ticker is green. The full reports with charts are in `Data Health.ipynb`.

In [ ]:
# Prices: fetch new data, refresh adjustment drift, verify a sample against Yahoo.
scraper = YF()
scraper.daily_update()

# Fundamentals: fetch SEC filings not fetched in the last 7 days, then rebuild the statement files.
Edgar().update()
EdgarStatements().build()

for title, health in (('PRICES', DataHealth()), ('FUNDAMENTALS', EdgarHealth())):
    scored = health.score()
    print(f"\n--- {title} ---")
    for key, value in health.summary(scored).items():
        print(f"{key}: {value}")
    display(health.problems(scored))

## 3. Maintenance (optional)
Uncomment a line to use it.

- `refresh_ticker` re-downloads one ticker's whole history and compares it with what is stored. It only reports unless `apply=True`, and it never removes stored rows.
- `force_refresh_all` does the same download for every watchlist ticker.
- `Edgar().update(force=True)` re-fetches SEC filings regardless of when they were last fetched.

In [ ]:
# scraper.refresh_ticker('SPY')               # report only
# scraper.refresh_ticker('SPY', apply=True)   # write the refreshed history
# scraper.daily_update(force_refresh_all=True)
# Edgar().update(force=True)